<a href="https://colab.research.google.com/github/Kommmi/Qaos/blob/main/05_Quantum_Algorithm_Local_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum Algorithm for Local Quantum Dynamics — Notebook 5

This notebook extends Notebooks 1–4 from state-vector calculations to **reconstruction from sampled measurements**. We implement the quantum kicked top in Qiskit, condition subsystem tomography on environment outcomes, and visualize the resulting local geometric quantum states.

The examples use an ideal Aer simulator with finite shots. Quantum circuits generate measurement counts; reconstruction and plotting are classical. Wasserstein, sensitivity, and coverage calculations are subsequent analyses using the earlier notebooks.

![Quantum–classical workflow](imagesREADME/QAlgo.jpg)

## 1. Setup

Run this notebook from the Qaos repository root. Install the optional Qiskit dependencies once with the cell below. In a fresh Colab session, first clone the repository and change into `Qaos`; do not delete an existing checkout. The new module must be uploaded to the repository before a fresh clone can use it.

In [ ]:
%pip install -e ".[qiskit]"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gqs.qiskit_qkt import (
    qkt_kick_circuit, show_single_experimental_step,
    measure_GQS_trajectory, plot_QS_history, ensemble_to_arrays,
)

## 2. Quantum kicked-top circuit

For $L$ qubits, $j=L/2$ and $\hbar=1$, one kick is

$$U_F=\exp[-i(\kappa/L)J_z^2]\exp[-i(\pi/2)J_y].$$

The circuit applies $R_y(\pi/2)$ on each qubit followed by $R_{ZZ}(\kappa/L)$ on every pair. Since $J_z^2=(L/4)I+(1/2)\sum_{a<b}Z_aZ_b$, this equals the Floquet map up to a global phase. All pair interactions commute, so this decomposition introduces **no Trotter error** for the specified kicked model.

Each qubit starts in $\cos(\theta_0/2)|0\rangle+e^{i\phi_0}\sin(\theta_0/2)|1\rangle$.

In [ ]:
theta0 = np.pi / 2 + 0.5
phi0 = np.pi / 2
nqubits = 3
kappa = 0.5
max_kicks = 20       # Increase to 100–200 for longer trajectories.
repetitions = 10000 # Shots per tomography basis, per kick.
seed = 42

## 3. Conditional measurement protocol

The system is qubit $q_0$; all other qubits form the environment. At each selected kick, separate circuit runs measure the system in $X$, $Y$, and $Z$, while the environment is measured in the computational basis. All measurements are terminal: each run starts from the same prepared initial state.

For environment outcome $e$, the conditional Bloch component is estimated as $(N_{e,0}-N_{e,1})/(N_{e,0}+N_{e,1})$ in each basis. Pooling environment counts across the three bases estimates its weight $\lambda_e$.

There are $3N$ circuits for $N$ kicks and $3N\times\text{repetitions}$ total shots. Although there are only three basis settings per kick, up to $2^{L-1}$ environment outcomes share these samples; rare outcomes can require many more shots.

In [ ]:
for basis in ("X", "Y", "Z"):
    print(f"System tomography: {basis}")
    show_single_experimental_step(
        nqubits=nqubits, kappa=kappa, theta0=theta0,
        phi0=phi0, basis=basis,
    )

## 4. Reconstruct the local ensemble

`Q_S[n-1]` contains the reconstructed ensemble after kick `n`; the initial state at kick zero is not included. Each record stores the environment label, probability `lambda`, raw Bloch components `x, y, z`, and angular coordinates `theta, phi`.

**Reconstruction assumption:** angular coordinates use the normalized Bloch direction, imposing purity. This is appropriate as an estimator for the ideal pure conditional states studied here. It must not be interpreted as mixed-state tomography for noisy hardware. Outcomes missing in any basis, or with zero Bloch direction, are omitted with a warning; check retained probability mass.

In [ ]:
Q_S = measure_GQS_trajectory(
    theta0=theta0, phi0=phi0, kappa=kappa,
    max_kicks=max_kicks, nqubits=nqubits,
    repetitions=repetitions, seed=seed,
)
retained_mass = np.array([sum(s["lambda"] for s in Q) for Q in Q_S])
print(f"Minimum retained mass: {retained_mass.min():.6f}")
Q_S[0][:2]

## 5. Visualize local dynamics

The plot displays the conditional states in $(\phi,\theta)$ coordinates, colored by probability. Earlier kicks use smaller, fainter markers. These angular coordinates are a visualization; ordinary Euclidean distance on this rectangle is not the Fubini–Study metric.

In [ ]:
fig, ax = plot_QS_history(Q_S, kick=max_kicks)
plt.show()

## 6. Compare interaction strengths and environment sizes

Repeat the same measurement protocol for three and four qubits at $\kappa=0,0.5,2.5$. Keep shots and initial-state angles fixed. Finite-shot scatter is present even in the noninteracting case; visual spread alone does not establish chaos.

In [ ]:
results = {}
for L in (3, 4):
    for coupling in (0.0, 0.5, 2.5):
        trajectory = measure_GQS_trajectory(
            theta0, phi0, coupling, max_kicks,
            nqubits=L, repetitions=repetitions, seed=seed,
        )
        results[L, coupling] = trajectory
        fig, ax = plot_QS_history(trajectory, kick=max_kicks)
        ax.set_title(f"L={L}, κ={coupling}")
        plt.show()

## 7. Connect to the Qaos diagnostics

Convert reconstructed pure-state directions into the `(chi_S, lambda_E)` arrays used by the existing plotting and optimal-transport functions. The adapter rejects incomplete probability mass by default; increasing shots is preferable to silently dropping outcomes.

Qiskit statevectors order qubits as $q_{L-1}\cdots q_0$, whereas Qaos tensor axes run left to right. When comparing directly with `Reduced_state_single_site`, reverse the statevector's tensor axes first to keep site labels aligned. Environment labels in this module follow $q_1,q_2,\ldots$.

The adapter enables later comparisons; this notebook does not yet estimate $\Gamma$ or the SSCI from measurement data.

In [ ]:
chi_S, lambda_E = ensemble_to_arrays(Q_S[-1])
print("Conditional states:", chi_S.shape)
print("Total probability:", lambda_E.sum())

## 8. Check against the exact Floquet operator

This small-system check compares the circuit with `gqs.dynamics`, allowing for the physically irrelevant global phase. The collective kicked-top operator is permutation-symmetric, so qubit-order reversal does not change this operator check.

In [ ]:
from qiskit.quantum_info import Operator
from gqs.dynamics import Hamiltonian_QK, floquet_operator_from_H

H1, H2 = Hamiltonian_QK(tau=1, kappa=kappa, nqubit=nqubits)
U_exact = floquet_operator_from_H(H1, H2, tau=1)
U_circuit = Operator(qkt_kick_circuit(nqubits, kappa)).data
phase = np.vdot(U_exact, U_circuit)
phase /= abs(phase)
error = np.max(np.abs(U_circuit - phase * U_exact))
print(f"Maximum operator error, up to global phase: {error:.3e}")
assert error < 1e-10

## Summary

We reconstruct local ensembles from ideal circuit measurement counts, then visualize how they change with interaction strength and environment size. The reusable implementation is in `gqs/qiskit_qkt.py`. Future noisy-hardware comparisons require conditional density-matrix reconstruction that retains mixedness, together with uncertainty estimates for the resulting diagnostics.